In [ ]:
import os, sys, subprocess

# ── Install dependencies ────────────────────────────────────
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    # Kaggle: use bundled wheels from aimo-3-utils (known working versions)
    _archive = '/kaggle/input/aimo-3-utils/wheels.tar.gz'
    _tmp = '/kaggle/tmp/setup'
    if not os.path.exists(_tmp):
        os.makedirs(_tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', _archive, '-C', _tmp], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--no-index', '--find-links', f'{_tmp}/wheels',
        'vllm', 'openai',
    ], check=True)
    # polars: install from PyPI (not in wheels)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'polars'], check=True)
else:
    # Colab/local: pin to known stable vLLM version
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'vllm==0.8.5', 'polars', 'openai', 'autoawq',
    ], check=True)

print('Dependencies installed.')

In [ ]:
import os, sys
from google.colab import userdata

ENV = 'kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') else \
      'colab'  if os.environ.get('COLAB_RELEASE_TAG')    else 'local'

print(f'Environment: {ENV}')

if ENV in ('colab', 'local'):
    os.environ["KAGGLE_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')

Environment: colab


In [ ]:
import os, sys, re, gc, time, math, subprocess, threading, queue
from typing import Optional
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl
from openai import OpenAI

# ── Model path + Test CSV ───────────────────────────────────
if ENV == 'kaggle':
    # H100 80GB: full BF16
    # Diagnose /kaggle/input/ structure
    import glob as _mglob
    print('=== /kaggle/input/ contents ===')
    if os.path.exists('/kaggle/input'):
        for _item in sorted(os.listdir('/kaggle/input')):
            _item_path = f'/kaggle/input/{_item}'
            try:
                _sub = os.listdir(_item_path)
                print(f'  {_item}/ -> {_sub[:5]}')
            except Exception:
                print(f'  {_item}')
    print('================================')

    # Dynamically find model path by searching for config.json
    _base = '/kaggle/input/deepseek-r1-distill-qwen-32b'
    _configs = _mglob.glob(f'{_base}/**/config.json', recursive=True)
    if _configs:
        MODEL_PATH = os.path.dirname(_configs[0])
        print(f'Found config.json -> MODEL_PATH: {MODEL_PATH}')
    else:
        # Fallback: search entire /kaggle/input for deepseek config
        _all_configs = _mglob.glob('/kaggle/input/**/config.json', recursive=True)
        _deepseek = [p for p in _all_configs if 'deepseek' in p.lower() or 'qwen' in p.lower()]
        if _deepseek:
            MODEL_PATH = os.path.dirname(_deepseek[0])
            print(f'Found config.json via broad search -> MODEL_PATH: {MODEL_PATH}')
        else:
            MODEL_PATH = f'{_base}/transformers/default/1'
            print(f'WARNING: config.json not found anywhere. All configs: {_all_configs}')
            print(f'Falling back to: {MODEL_PATH}')
    QUANTIZATION = None
    DTYPE = 'bfloat16'
    TEST_CSV = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv'

else:
    # Colab A100 40GB: AWQ 4-bit (~16GB VRAM)
    MODEL_PATH = 'casperhansen/deepseek-r1-distill-qwen-32b-awq'
    QUANTIZATION = 'awq'
    DTYPE = 'float16'

    # Install kagglehub for dataset downloads
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True)
    import kagglehub

    # Download test.csv from competition
    _comp_dir = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3'
    if os.path.exists(f'{_comp_dir}/test.csv'):
        TEST_CSV = f'{_comp_dir}/test.csv'
        print(f'Test CSV (cached): {TEST_CSV}')
    else:
        try:
            data_path = kagglehub.competition_download('ai-mathematical-olympiad-progress-prize-3')
            TEST_CSV = f'{data_path}/test.csv'
        except Exception as e:
            print(f'competition_download failed: {e}')
            TEST_CSV = '/content/test.csv'
            with open(TEST_CSV, 'w') as _f:
                _f.write('"id","problem"\n')
                _f.write('"000aaa","What is $1-1$?"\n')
                _f.write('"111bbb","What is $0\\times10$?"\n')
                _f.write('"222ccc","Solve $4+x=4$ for $x$."\n')
            print(f'Test CSV created inline: {TEST_CSV}')

    if not os.path.exists(TEST_CSV):
        TEST_CSV = '/content/test.csv'
        with open(TEST_CSV, 'w') as _f:
            _f.write('"id","problem"\n')
            _f.write('"000aaa","What is $1-1$?"\n')
            _f.write('"111bbb","What is $0\\times10$?"\n')
            _f.write('"222ccc","Solve $4+x=4$ for $x$."\n')
        print(f'Test CSV created (fallback): {TEST_CSV}')

    # Download 50problems dataset
    try:
        problems_path = kagglehub.dataset_download('amanatar/50problems')
        print(f'50problems path: {problems_path}')
    except Exception as e:
        print(f'50problems download failed: {e}')

print(f'MODEL_PATH: {MODEL_PATH}')
print(f'QUANTIZATION: {QUANTIZATION}, DTYPE: {DTYPE}')
print(f'TEST_CSV: {TEST_CSV}')


class CFG:

    ANSWER_ONLY_PROMPT = (
        'You are an expert in solving math olympiad problems.\n'
        'Think carefully, then give your final answer.\n'
        'The answer must be a non-negative integer between 0 and 99999.\n'
        'End with: ANSWER: [integer]'
    )

    TIR_PROMPT = (
        'You are an expert in solving math olympiad problems.\n'
        'Write and execute Python code snippets in your thinking to verify calculations.\n'
        'Start with small cases or brute force, then generalise.\n'
        'Use sympy for exact arithmetic. Always print intermediate results.\n'
        'The answer must be a non-negative integer between 0 and 99999.\n'
        'End with: ANSWER: [integer]'
    )

    PURE_MATH_PROMPT = (
        'You are an IMO gold medallist solving this problem using pure mathematical reasoning only.\n'
        '1. CLASSIFY the mathematical domain (number theory, combinatorics, geometry, algebra).\n'
        '2. STATE any relevant theorems or lemmas before applying them.\n'
        '3. BUILD a rigorous step-by-step proof. Every claim must be justified.\n'
        '4. VERIFY the answer satisfies every constraint in the problem.\n'
        'Do NOT write or execute code.\n'
        'The answer must be a non-negative integer between 0 and 99999.\n'
        'End with: ANSWER: [integer]'
    )

    WORKING_BACKWARDS_PROMPT = (
        'You are an expert mathematician. Solve this problem by working backwards.\n'
        '1. BOUND: Identify constraints the answer must satisfy (parity, modular conditions, size).\n'
        '2. HYPOTHESISE: Assume the answer equals N satisfying those constraints.\n'
        '3. ELIMINATE: Rule out impossible values of N using modular arithmetic, parity, or size arguments.\n'
        '4. CONFIRM: Prove the surviving candidate is correct.\n'
        'You may use code to test specific candidate values.\n'
        'The answer must be a non-negative integer between 0 and 99999.\n'
        'End with: ANSWER: [integer]'
    )

    # Attempt config: (prompt_attr, temperature)
    PHASE2_ATTEMPTS = [
        ('TIR_PROMPT',              0.3),
        ('TIR_PROMPT',              0.7),
        ('PURE_MATH_PROMPT',        0.5),
        ('PURE_MATH_PROMPT',        0.9),
        ('WORKING_BACKWARDS_PROMPT', 0.4),
        ('WORKING_BACKWARDS_PROMPT', 0.8),
    ]

    served_model_name = 'deepseek-r1-32b'
    model_path = MODEL_PATH
    quantization = QUANTIZATION
    dtype = DTYPE

    max_model_len = 32768
    gpu_memory_utilization = 0.92
    max_num_seqs = 32

    phase1_max_tokens = 4096
    phase2_max_tokens_kaggle = 16384
    phase2_max_tokens_colab = 8192

    notebook_limit = 17400
    server_timeout = 300
    session_timeout = 960
    early_stop = 4
    seed = 42

competition_download failed: User is not authenticated
Test CSV created inline: /content/test.csv


100%|██████████| 4.55k/4.55k [00:00<00:00, 10.4MB/s]

Extracting files...
50problems path: /root/.cache/kagglehub/datasets/amanatar/50problems/versions/1
MODEL_PATH: casperhansen/deepseek-r1-distill-qwen-32b-awq
QUANTIZATION: awq, DTYPE: float16
TEST_CSV: /content/test.csv


In [ ]:
def start_vllm_server():
    cmd = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', CFG.model_path,
        '--served-model-name', CFG.served_model_name,
        '--dtype', CFG.dtype,
        '--kv-cache-dtype', 'auto',
        '--tensor-parallel-size', '1',
        '--max-model-len', str(CFG.max_model_len),
        '--gpu-memory-utilization', str(CFG.gpu_memory_utilization),
        '--max-num-seqs', str(CFG.max_num_seqs),
        '--enable-prefix-caching',
        '--disable-log-stats',
        '--host', '0.0.0.0',
        '--port', '8000',
    ]
    if CFG.quantization:
        cmd += ['--quantization', CFG.quantization]

    # Ensure CUDA device is visible to the subprocess
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = '0'

    print(f'vLLM cmd: {" ".join(cmd)}')
    log_file = open('vllm_server.log', 'w')
    proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT,
                            start_new_session=True, env=env)
    return proc, log_file


def wait_for_server(proc, log_file, client, timeout=CFG.server_timeout):
    print('Waiting for vLLM server...')
    start = time.time()
    for _ in range(timeout):
        rc = proc.poll()
        if rc is not None:
            log_file.flush()
            with open('vllm_server.log', 'r') as f:
                logs = f.read()
            raise RuntimeError(f'Server died with code {rc}. Logs:\n{logs}')
        try:
            client.models.list()
            print(f'Server ready in {time.time() - start:.1f}s')
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError('Server startup timeout')


os.environ['CUDA_VISIBLE_DEVICES'] = '0'

client = OpenAI(base_url='http://0.0.0.0:8000/v1', api_key='sk-local', timeout=CFG.session_timeout)
server_proc, server_log = start_vllm_server()
wait_for_server(server_proc, server_log, client)

vLLM cmd: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model casperhansen/deepseek-r1-distill-qwen-32b-awq --served-model-name deepseek-r1-32b --dtype float16 --kv-cache-dtype auto --tensor-parallel-size 1 --max-model-len 32768 --gpu-memory-utilization 0.92 --max-num-seqs 32 --enable-prefix-caching --disable-log-stats --host 0.0.0.0 --port 8000 --quantization awq
Waiting for vLLM server...
Server ready in 0.0s


In [ ]:
def extract_answer(text: str) -> int | None:
    # Prefer text after </think> to avoid picking up intermediate answers
    after_think = text.split('</think>')[-1] if '</think>' in text else text

    # Priority 1: ANSWER: [integer] (last occurrence)
    matches = re.findall(r'ANSWER:\s*([0-9,]+)', after_think, re.IGNORECASE)
    if matches:
        value = int(matches[-1].replace(',', ''))
        return value % 100000

    # Priority 2: \boxed{integer} (last occurrence)
    matches = re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', after_think)
    if matches:
        value = int(matches[-1].replace(',', ''))
        return value % 100000

    # Priority 3: \boxed{} in full text (fallback for missing </think>)
    matches = re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text)
    if matches:
        value = int(matches[-1].replace(',', ''))
        return value % 100000

    return None

In [ ]:
def select_answer(answers: list[int | None]) -> int:
    valid = [a for a in answers if a is not None]
    if not valid:
        return 0

    counts = Counter(valid)
    most_common, count = counts.most_common(1)[0]
    return most_common

In [ ]:
notebook_start_time = time.time()
problems_remaining = 50


def _call_model(problem: str, system_prompt: str, temperature: float, max_tokens: int) -> str | None:
    """Single LLM call. Returns full response text or None on failure."""
    user_content = f'Problem: {problem}\n\n<think>\n'
    try:
        response = client.chat.completions.create(
            model=CFG.served_model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_content},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
            seed=CFG.seed,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f'  [ERROR] {e}')
        return None


def _run_attempt(problem, system_prompt, temperature, max_tokens, stop_event):
    """Run a single attempt; respects stop_event for early stopping."""
    if stop_event.is_set():
        return None
    text = _call_model(problem, system_prompt, temperature, max_tokens)
    if text is None:
        return None
    return extract_answer(text)


def solve_problem(problem: str) -> int:
    global problems_remaining

    print(f'\nProblem: {problem[:100]}...')

    elapsed = time.time() - notebook_start_time
    time_left = CFG.notebook_limit - elapsed
    budget_per_problem = time_left / max(problems_remaining, 1)

    # ── Phase 1: Quick screening (greedy x2) ────────────────
    phase1_answers = []
    for _ in range(2):
        text = _call_model(problem, CFG.ANSWER_ONLY_PROMPT, temperature=0.0, max_tokens=CFG.phase1_max_tokens)
        if text is not None:
            ans = extract_answer(text)
            phase1_answers.append(ans)

    valid_p1 = [a for a in phase1_answers if a is not None]
    if len(valid_p1) == 2 and valid_p1[0] == valid_p1[1]:
        print(f'  Phase 1 consensus: {valid_p1[0]}')
        problems_remaining = max(0, problems_remaining - 1)
        return valid_p1[0]

    # ── Time budget check ───────────────────────────────────
    if budget_per_problem < 60:
        print('  Time pressure: returning Phase 1 result')
        answer = select_answer(phase1_answers)
        problems_remaining = max(0, problems_remaining - 1)
        return answer

    # ── Phase 2: Deep exploration ───────────────────────────
    max_tokens_p2 = CFG.phase2_max_tokens_kaggle if ENV == 'kaggle' else CFG.phase2_max_tokens_colab

    attempts_config = CFG.PHASE2_ATTEMPTS
    if budget_per_problem < 180:
        attempts_config = attempts_config[:3]  # Halve attempts

    all_answers = list(phase1_answers)  # Carry Phase 1 answers forward
    stop_event = threading.Event()

    with ThreadPoolExecutor(max_workers=len(attempts_config)) as executor:
        futures = {}
        for prompt_attr, temp in attempts_config:
            prompt = getattr(CFG, prompt_attr)
            f = executor.submit(_run_attempt, problem, prompt, temp, max_tokens_p2, stop_event)
            futures[f] = (prompt_attr, temp)

        for future in as_completed(futures):
            ans = future.result()
            all_answers.append(ans)

            # Early stopping: majority reached
            valid = [a for a in all_answers if a is not None]
            if valid:
                counts = Counter(valid)
                top_ans, top_count = counts.most_common(1)[0]
                total = len(phase1_answers) + len(attempts_config)
                if top_count >= total // 2 + 1:
                    stop_event.set()
                    break

    answer = select_answer(all_answers)
    print(f'  Final answer: {answer} (from {all_answers})')
    problems_remaining = max(0, problems_remaining - 1)
    return answer

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_value = id_.item(0)
    question_text = question.item(0)

    gc.disable()
    final_answer = solve_problem(question_text)
    gc.enable()
    gc.collect()

    return pl.DataFrame({'id': id_value, 'answer': final_answer})


if ENV == 'kaggle':
    import kaggle_evaluation.aimo_3_inference_server
    inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        inference_server.serve()
    else:
        inference_server.run_local_gateway((TEST_CSV,))
else:
    test_df = pl.read_csv(TEST_CSV)
    results = []
    for row in test_df.iter_rows(named=True):
        answer = solve_problem(row['problem'])
        results.append({'id': row['id'], 'answer': answer})
        print(f"[{row['id']}] {row['problem'][:60]} -> {answer}")
    result_df = pl.DataFrame(results)
    print(result_df)


Problem: What is $1-1$?...
  Phase 1 consensus: 0
[000aaa] What is $1-1$? -> 0

Problem: What is $0\times10$?...
  Phase 1 consensus: 0
[111bbb] What is $0\times10$? -> 0

Problem: Solve $4+x=4$ for $x$....
  Phase 1 consensus: 0
[222ccc] Solve $4+x=4$ for $x$. -> 0
shape: (3, 2)
┌────────┬────────┐
│ id     ┆ answer │
│ ---    ┆ ---    │
│ str    ┆ i64    │
╞════════╪════════╡
│ 000aaa ┆ 0      │
│ 111bbb ┆ 0      │
│ 222ccc ┆ 0      │
└────────┴────────┘


In [ ]:
import polars as pl
import pandas as pd
import os, time, json as _json, glob as _glob
from pathlib import Path
from collections import Counter

# ── Find 50problems.csv ───────────────────────────────────
_glob_results = _glob.glob('/kaggle/input/**/50problems.csv', recursive=True)
print(f'/kaggle/input contents: {os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "N/A"}')
print(f'glob results: {_glob_results}')
_candidates = [
    '/kaggle/input/50problems/50problems.csv',
    *_glob_results,
    *_glob.glob('/root/.cache/kagglehub/datasets/amanatar/50problems/**/50problems.csv', recursive=True),
    'data/50problems/50problems.csv',
]
FILE_PATH = next((p for p in _candidates if os.path.exists(p)), None)
print(f'FILE_PATH: {FILE_PATH}')

if FILE_PATH is None:
    raise FileNotFoundError('50problems.csv not found. Check dataset_sources in kernel-metadata.json.')

# ── Output dir ────────────────────────────────────────────
SAVE_DIR = Path('/kaggle/working/eval_output')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH  = SAVE_DIR / 'eval_results.csv'
JSON_PATH = SAVE_DIR / 'eval_results.json'

# ── Load data ─────────────────────────────────────────────
external_df = pd.read_csv(FILE_PATH)
print(f'\nEvaluation start: {len(external_df)} problems')
print(f'Results will be saved to: {SAVE_DIR}\n')

# ── Eval loop ─────────────────────────────────────────────
records = []
correct_count = 0

for idx, row in external_df.iterrows():
    problem_text   = str(row['Problem'])
    correct_answer = int(row['Answer'])
    print(f'--- Problem {idx+1}/{len(external_df)} ---')

    start = time.time()
    predicted = solve_problem(problem_text)
    elapsed = time.time() - start

    # Failure type classification
    if predicted == correct_answer:
        failure_type = None
        correct_count += 1
    else:
        failure_type = 'type_b'  # Simplified: no detailed attempt tracking here

    mark = 'OK' if predicted == correct_answer else f'FAIL ({failure_type})'
    print(f'Answer: {correct_answer} | Predicted: {predicted} | {mark} | {elapsed:.1f}s')

    record = {
        'idx':            idx,
        'problem':        problem_text,
        'correct_answer': correct_answer,
        'predicted':      predicted,
        'is_correct':     predicted == correct_answer,
        'failure_type':   failure_type,
        'all_answers':    [],
        'n_attempts':     8,
        'n_valid':        0,
        'time_sec':       round(elapsed, 1),
        'problem_len':    len(problem_text),
    }
    records.append(record)

    # Save after each problem (session disconnect protection)
    pd.DataFrame(records).to_csv(CSV_PATH, index=False)
    with open(JSON_PATH, 'w') as _f:
        _json.dump(records, _f, ensure_ascii=False, indent=2)
    print(f'  -> saved ({len(records)}/{len(external_df)} done)')

# ── Summary ───────────────────────────────────────────────
results_df = pd.DataFrame(records)
print(f'\n{"="*60}')
print(f'Result: {correct_count}/{len(external_df)} correct ({correct_count/len(external_df)*100:.1f}%)')

failed = results_df[~results_df['is_correct']]
if len(failed):
    print(f'Failed indices: {failed["idx"].tolist()}')
    print(f'Failure types: {failed["failure_type"].value_counts().to_dict()}')
print(f'{"="*60}')
print(f'\nDownload results from: Output tab -> eval_output/eval_results.csv')

/kaggle/input contents: []
glob results: []
FILE_PATH: /root/.cache/kagglehub/datasets/amanatar/50problems/versions/1/50problems.csv

Evaluation start: 50 problems
Results will be saved to: /kaggle/working/eval_output

--- Problem 1/50 ---

Problem: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie ...
  [ERROR] Request timed out.
